# Demand Forecasting Kaggle Quick Runner
Use this notebook to run the project on Kaggle with minimal setup.

Attach 3 Kaggle datasets before running:
1. Code dataset
2. M5 dataset
3. Walmart dataset


In [ ]:
import shlex
import subprocess
from pathlib import Path

CODE_DATASET = 'your-code-dataset'
M5_DATASET = 'your-m5-dataset'
WALMART_DATASET = 'your-walmart-dataset'

PROJECT_DIR = Path('/kaggle/working/DS Demand Forecasting')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_DIR


In [ ]:
def run_cmd(command: str):
    subprocess.run(command, shell=True, check=True)

def copy_or_unzip(dataset_name: str, expected_folder: str, zip_name: str):
    base = Path('/kaggle/input') / dataset_name
    folder_path = base / expected_folder
    zip_path = base / zip_name

    if folder_path.exists():
        run_cmd(f"cp -r {shlex.quote(str(folder_path))} {shlex.quote(str(PROJECT_DIR))}")
        return

    if zip_path.exists():
        run_cmd(f"unzip -o {shlex.quote(str(zip_path))} -d {shlex.quote(str(PROJECT_DIR))}")
        return

    raise FileNotFoundError(f'Could not find {expected_folder} or {zip_name} in {base}')

code_base = Path('/kaggle/input') / CODE_DATASET
code_zip = code_base / 'project_code.zip'
if code_zip.exists():
    run_cmd(f"unzip -o {shlex.quote(str(code_zip))} -d {shlex.quote(str(PROJECT_DIR))}")
else:
    run_cmd(f"cp -r {shlex.quote(str(code_base))}/* {shlex.quote(str(PROJECT_DIR))}")

copy_or_unzip(M5_DATASET, 'm5-forecasting-accuracy', 'm5_data.zip')
copy_or_unzip(WALMART_DATASET, 'archive', 'walmart_data.zip')

print('Project directory:', PROJECT_DIR)
for p in sorted(PROJECT_DIR.iterdir()):
    print('-', p.name)


In [ ]:
%cd '/kaggle/working/DS Demand Forecasting'
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt


In [ ]:
for d in [
    'outputs/metrics',
    'outputs/checkpoints',
    'outputs/figures',
    'outputs/reports',
    'data/processed',
    'data/predictions',
]:
    Path(d).mkdir(parents=True, exist_ok=True)
print('Output directories ready')


## Run Pipeline


In [ ]:
!python scripts/run_data_pipeline.py


In [ ]:
RUN_LIGHTGBM = True
RUN_SARIMA = False
RUN_PROPHET = False

if RUN_LIGHTGBM:
    run_cmd("python -c \"from src.models.lightgbm import train_and_predict; train_and_predict()\"")

if RUN_SARIMA:
    run_cmd("python -c \"from src.models.sarima import train_and_predict; train_and_predict()\"")

if RUN_PROPHET:
    run_cmd("python -c \"from src.models.prophet import train_and_predict; train_and_predict()\"")

print('Model run complete')


In [ ]:
!python scripts/run_evaluation.py


In [ ]:
import pandas as pd

metrics = pd.read_csv('outputs/metrics/model_comparison.csv')
impact = pd.read_csv('outputs/metrics/business_impact.csv')
display(metrics)
display(impact)


In [ ]:
!zip -r outputs_bundle.zip outputs data/predictions data/processed/split_metadata.json
print('Created outputs_bundle.zip in /kaggle/working')
